In [1]:
# pip install tavily-python

In [2]:
import os
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langgraph.graph import StateGraph, END
from typing import TypedDict, Any, Annotated
from langchain.tools import tool
from tavily import TavilyClient
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import SystemMessage

# (not needed) -
# from langchain.agents import create_agent : - not needed anymore as that's the old LangChain agent pattern, LangGraph replaces it entirely
# from langchain_core.documents import Document - not needed (ingestion is already done, 677 chunks are in DB)
# from langchain_text_splitters import RecursiveCharacterTextSplitter - not needed (ingestion is already done, 677 chunks are in DB)
# import fitz - not needed (ingestion is already done, 677 chunks are in DB)
# import ast :- You'd need this if you were evaluating or transforming Python code at runtime. The agent just retrieves text and calls an LLM.
# import numpy as np : — numerical array operations, matrix math. ChromaDB handles all the vector similarity math internally. You never touch raw vectors in this project.
# import json : - — serializing/deserializing JSON. You'd need this if you were manually parsing LLM responses into structured data. But since you're using Pydantic schemas and LangChain's tool framework, that's handled automatically.
# import re : - The LLM handles language understanding, and ChromaDB handles search. No string pattern matching needed.

In [3]:
# llm= ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [4]:
# llm_with_tools = llm.bind_tools([search_studies, compare_studies, web_search])

In [5]:
class AgentState(TypedDict):
    query: str
    messages: Annotated[list, add_messages]   
    final_result: str
    sources : str
    # tasks : list   
    # retries: int
    # valid: bool

In [6]:
vector_store = Chroma(
    persist_directory="C:/Users/Pranali Jadhav/OneDrive/Documents/GEN_AI/Learnbay/clinical_research_ai_assistant/clinical_research_ai_assistant_langgraph/research_asst_db",
    embedding_function=OpenAIEmbeddings(model='text-embedding-3-small')
)
print(vector_store._collection.count())

C:\Users\Pranali Jadhav\AppData\Local\Temp\ipykernel_21516\3875359495.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


677


In [9]:
# #test a raw retrieval before touching any LangGraph code:---

# results = vector_store.similarity_search("what is the mortality rate", k=3)
# for r in results:
#     print(r.metadata)
#     print(r.page_content[:200])
#     print("---")

In [10]:
# Convert vector_store to a retriever — this is what the tools will actually use:-------
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

In [12]:
# test
# quick test of retriever :-------
# docs = retriever.invoke("mortality rate salt substitutes")
# print(f"Got {len(docs)} chunks")
# print(docs[0].metadata)

#singel doc check:-
# docs[0] #-- full doc details
# docs[0].metadata #-- only metadata

#all 5 chosen by search_kwargs
# for i in docs:
#     print(i.metadata)

In [13]:
@tool
def search_studies(query: str) ->list:
    '''Search the clinical research corpus for studies relevant to a query.'''
    print("search_studies agent invoked")
    docs = retriever.invoke(query) # -- it's LangChain's built in method
    print(f"Docs retrieved: {len(docs)}")
    result = []
    for doc in docs:
        meta = doc.metadata
        result.append({
            "study_id": meta.get("study_id"),
            "author_year": meta.get("author_year"),
            "page_number": meta.get("page_number"),
            "content": doc.page_content[:500]

        })
    return result

In [14]:
# test search_studies()
# search_studies.invoke("mortality rate salt substitutes")

In [15]:
# test -
# similar_doc_retrive = vector_store.similarity_search("mortality rate", k=5, filter={"study_id": "1"})
# # print(similar_doc_retrive)

In [17]:
@tool
def compare_studies(query: str, ids: list) -> list:
    """Search the clinical research corpus for studies relevant to a query for comparison"""
    print("compare_studies agent invoked")
    result = []
    try:
        for id in ids:
            str_id = str(id)
            print(f"Searching for id: {str_id}, type: {type(str_id)}")
            docs = vector_store.similarity_search(query, k=3, filter={"study_id": str_id})
            print(f"Docs found: {len(docs)}")
            for doc in docs:
                meta = doc.metadata
                result.append({
                    "study_id": meta.get("study_id"),
                    "author_year": meta.get("author_year"),
                    "page_number": meta.get("page_number"),
                    "content": doc.page_content[:500]
                })
    except Exception as e:
        print(f"Error: {e}")
    return result

In [18]:
# test -
# 3 each id thats 6 in total for id 1 and 2 = 6 docs
# compare_studies.invoke({"query": "mortality rate", "ids": ["1", "2"]})

In [19]:
tavily = TavilyClient(api_key="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")
results = tavily.search("mortality rate")
print (results, end="\n")

{'query': 'mortality rate', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.scitechnol.com/peer-review/understanding-mortality-rate-a-comprehensive-guide-Evhb.php?article_id=22101', 'title': 'Understanding Mortality Rate: A Comprehensive Guide - SciTechnol', 'content': 'Department of Clinical and Experimental Medicine, University of Pisa, Pisa, Italy. **Received date:** 21 February, 2023, Manuscript No. IJGH-23-93529;. Mortality rate refers to the number of deaths that occur within a population during a specific time period. It is an important indicator of the health of a population and is often used to assess the effectiveness of healthcare interventions. We will also examine the factors that influence mortality rate and its significance in public health. Crude mortality rate refers to the number of deaths in a population divided by the total population during a given period. Agespecific mortality rate is the number of deaths within a specif

In [20]:
@tool
def web_search(query : str)->list:
    """Search the details on the web as per queary by user"""
    print("web_search agent invoked")
    result = []
    docs = tavily.search(query)
    for doc in docs['results']: # this loop will fetch details for each id comes from above loop
            # meta = doc.metadata
            result.append({
                "URL": doc.get("url"),
                "Title": doc.get("title"),
                # "content": meta.get("page_number"),
                "content": doc.get("content")
            })
    return result

In [21]:
# test
# web_search.invoke("mortality rate?")

In [22]:
llm= ChatOpenAI(model='gpt-4o-mini', temperature=0)
llm_with_tools = llm.bind_tools([search_studies, compare_studies, web_search])

In [23]:
# call_model — for the LLM decision node
# from langchain_core.messages import SystemMessage

def call_model(state: AgentState):
    system = SystemMessage(content="""You are a Clinical Research Assistant.

TOOL USAGE RULES (follow in order):
1. If the query is about guidelines, policies, or recent developments → call web_search first
2. For all other clinical questions → call search_studies first
3. If asked to compare specific studies → call compare_studies
4. Always use the exact user query when calling any tool
5. Always cite Study ID and Author/Year in your final answer""")
    messages = [system] + state["messages"]
    print(f"Total messages going to LLM: {len(messages)}")
    print(f"First message type: {type(messages[0])}")
    response = llm_with_tools.invoke(messages)
    print(f"Tool calls in response: {response.tool_calls}")
    return {"messages": [response]}

In [24]:
#call_tools — for the tool executor node
#That's your entire second node — no function needed.
call_tools = ToolNode([search_studies, compare_studies, web_search]) 

In [25]:
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return "end"

In [ ]:
graph = StateGraph(AgentState)
graph.add_node('LLM', call_model)
graph.add_node('tools',call_tools)
graph.set_entry_point('LLM')
graph.add_edge('tools', 'LLM') # --means — after tools finishes executing, always go back to LLM. No condition, no decision, just always.add_edge creates a fixed, unconditional connection between two nodes.
graph.add_conditional_edges(
    'LLM',
    should_continue,
    {
        'tools': 'tools',
        'end': END
    }
)

app = graph.compile()

In [ ]:
# # test- search_studies agent
# result = app.invoke({"messages": [{"role": "user", "content": "what is the mortality rate?"}], "query": "what is the mortality rate?", "sources": [], "final_answer": ""})

Total messages going to LLM: 2
First message type: <class 'langchain_core.messages.system.SystemMessage'>
Tool calls in response: [{'name': 'web_search', 'args': {'query': 'mortality rate definition and statistics'}, 'id': 'call_xmsgiYSCMgZh5S5s6kyzqDcI', 'type': 'tool_call'}]
web_search agent invoked
Total messages going to LLM: 4
First message type: <class 'langchain_core.messages.system.SystemMessage'>
Tool calls in response: []


In [ ]:
# # test- compare_studies agent
# result = app.invoke({
#     "messages": [{"role": "user", "content": "compare study 1 and study 2 on mortality rate"}],
#     "query": "compare study 1 and study 2 on mortality rate",
#     "sources": [],
#     "final_answer": ""
# })

Total messages going to LLM: 2
First message type: <class 'langchain_core.messages.system.SystemMessage'>
Tool calls in response: [{'name': 'search_studies', 'args': {'query': 'mortality rate'}, 'id': 'call_PGuHkyXjcce0ZC7t0C0Z0l1n', 'type': 'tool_call'}]
search_studies agent invoked
Docs retrieved: 5
Total messages going to LLM: 4
First message type: <class 'langchain_core.messages.system.SystemMessage'>
Tool calls in response: [{'name': 'compare_studies', 'args': {'query': 'mortality rate', 'ids': ['1', '2']}, 'id': 'call_iGRPT0TpT4s8t7sC0A1vO2fZ', 'type': 'tool_call'}]
compare_studies agent invoked
Searching for id: 1, type: <class 'str'>
Docs found: 3
Searching for id: 2, type: <class 'str'>
Docs found: 3
Total messages going to LLM: 6
First message type: <class 'langchain_core.messages.system.SystemMessage'>
Tool calls in response: []


In [ ]:
# print(result['messages'][-1].content)

The mortality rate can vary based on different factors and populations. According to a study by Honghao (2025), the all-cause mortality rate was estimated to be:

- **Intermediate risk**: 62 per 1000 persons
- **High risk**: 143 per 1000 persons

For cardiovascular mortality, the rates were:

- **Intermediate risk**: 29 per 1000 persons
- **High risk**: 92 per 1000 persons

These estimates reflect the mortality rates in specific populations and conditions. For more detailed information, you can refer to Study ID 1 by Honghao (2025).
